In [ ]:
!pip install docling-core[chunking]
!pip install docling-core[chunking-openai]
!pip install docling
!pip install -qU pip docling transformers

In [1]:
def pretty(obj):
  import json
  print(json.dumps(obj.__dict__,indent=2,default=str))

In [14]:
import logging
from pathlib import Path

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    ThreadedPdfPipelineOptions,
    TableFormerMode,
    PdfPipelineOptions
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.layout_model_specs import DOCLING_LAYOUT_HERON_101
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions

# Configure accelerator options for GPU


_log = logging.getLogger(__name__)

# Faster rendering
IMAGE_RESOLUTION_SCALE = 2.0

input_doc_path = r"C:\Users\hp\OneDrive\Desktop\rag_learning_materials\Vanilla_document_based_rag\JudgeFlow.pdf"
output_dir = Path("/content/drive/MyDrive/scratch")


def main():
    logging.basicConfig(level=logging.INFO)

    # Use threaded options (important)
    pipeline_options = PdfPipelineOptions()

    # ---------- OCR ----------
    pipeline_options.do_ocr = False  # Fast if PDFs are digital
    pipeline_options.force_backend_text = True #Backend = text source, Layout model = page structure then organises the text into those structures, preserving reading order, table cells, etc.

    # ---------- Layout ----------
    pipeline_options.layout_options.model_spec = DOCLING_LAYOUT_HERON_101

    # ---------- Tables (fast mode) ----------
    pipeline_options.do_table_structure = True
    pipeline_options.table_structure_options.do_cell_matching = False
    pipeline_options.table_structure_options.mode = TableFormerMode.FAST

    # ---------- Images (disable for speed) ----------
    #pipeline_options.images_scale = IMAGE_RESOLUTION_SCALE
    pipeline_options.generate_page_images = False
    pipeline_options.generate_picture_images = False
    # ---------- Disable enrichment (prevents heavy backend loading) ----------
    pipeline_options.do_formula_enrichment = False
    pipeline_options.do_code_enrichment = False
    pipeline_options.do_picture_description = False
    pipeline_options.do_picture_classification = False

    # ---------- Memory optimization ----------
    pipeline_options.generate_parsed_pages = False

    # ---------- Thread tuning (optional speed boost) ----------

    # Create converter
    doc_converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )

    return doc_converter

In [15]:
doc_converter = main()

In [16]:
import time
start_time = time.time()
doc_converter.initialize_pipeline(InputFormat.PDF)
init_runtime = time.time() - start_time
_log.info(f"Pipeline initialized in {init_runtime:.2f} seconds.")

INFO:docling.document_converter:Initializing pipeline for StandardPdfPipeline with options hash 2f8fe76d8bcdc9a5e972172b3a132a0e
INFO:docling.utils.accelerator_utils:Accelerator device: 'cpu'
INFO:docling.utils.accelerator_utils:Accelerator device: 'cpu'
INFO:__main__:Pipeline initialized in 1.75 seconds.


In [17]:
start_time = time.time()
conv_res = doc_converter.convert(input_doc_path)

INFO:docling.datamodel.document:detected formats: [<InputFormat.PDF: 'pdf'>]
INFO:docling.document_converter:Going to convert document batch...
INFO:docling.document_converter:Initializing pipeline for StandardPdfPipeline with options hash 601f7c909b822cebd4d7486c387482eb
INFO:docling.utils.accelerator_utils:Accelerator device: 'cpu'
INFO:docling.utils.accelerator_utils:Accelerator device: 'cpu'
INFO:docling.pipeline.base_pipeline:Processing document JudgeFlow.pdf
ERROR:docling.pipeline.standard_pdf_pipeline:Stage preprocess failed for run 1, pages [11]: std::bad_alloc
ERROR:docling.pipeline.standard_pdf_pipeline:Stage preprocess failed for run 1, pages [12]: std::bad_alloc
ERROR:docling.pipeline.standard_pdf_pipeline:Stage preprocess failed for run 1, pages [13]: std::bad_alloc
ERROR:docling.pipeline.standard_pdf_pipeline:Stage preprocess failed for run 1, pages [14]: std::bad_alloc
ERROR:docling.pipeline.standard_pdf_pipeline:Stage preprocess failed for run 1, pages [15]: std::bad_al

In [ ]:
import json
print(json.dumps(conv_res.document.__dict__, indent=2,default=str))

{
  "schema_name": "DoclingDocument",
  "version": "1.9.0",
  "name": "JudgeFlow",
  "origin": "mimetype='application/pdf' binary_hash=12268160804681667642 filename='JudgeFlow.pdf' uri=None",
  "furniture": "self_ref='#/furniture' parent=None children=[] content_layer=<ContentLayer.FURNITURE: 'furniture'> meta=None name='_root_' label=<GroupLabel.UNSPECIFIED: 'unspecified'>",
  "body": "self_ref='#/body' parent=None children=[RefItem(cref='#/texts/0'), RefItem(cref='#/texts/1'), RefItem(cref='#/texts/2'), RefItem(cref='#/texts/3'), RefItem(cref='#/texts/4'), RefItem(cref='#/texts/5'), RefItem(cref='#/texts/6'), RefItem(cref='#/texts/7'), RefItem(cref='#/texts/8'), RefItem(cref='#/texts/9'), RefItem(cref='#/texts/10'), RefItem(cref='#/texts/11'), RefItem(cref='#/texts/12'), RefItem(cref='#/texts/13'), RefItem(cref='#/texts/14'), RefItem(cref='#/groups/0'), RefItem(cref='#/texts/19'), RefItem(cref='#/pictures/0'), RefItem(cref='#/texts/35'), RefItem(cref='#/texts/36'), RefItem(cref='#/te

In [18]:
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from transformers import AutoTokenizer

from docling.chunking import HybridChunker

EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
MAX_TOKENS = 500  # set to a small number for illustrative purposes

tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained(EMBED_MODEL_ID),
    max_tokens=MAX_TOKENS,  # optional, by default derived from `tokenizer` for HF case
)

In [19]:
#Document stores data → Serializer decides how to present it
from docling_core.transforms.chunker.hierarchical_chunker import (
    ChunkingDocSerializer,
    ChunkingSerializerProvider,
    TripletTableSerializer
)
from docling_core.transforms.serializer.markdown import MarkdownParams



class MDTableSerializerProvider(ChunkingSerializerProvider):
    def get_serializer(self, doc):
        return ChunkingDocSerializer(
            doc=doc,
            params=MarkdownParams(
                image_placeholder="<!-- image -->"
            ),
            #table_serializer=TripletTableSerializer(),  # configuring a different table serializer
        )
    
chunker = HybridChunker(
    tokenizer=tokenizer,
    serializer_provider=MDTableSerializerProvider(),
    merge_peers=True,  # optional, defaults to True
)
chunk_iter = chunker.chunk(dl_doc=conv_res.document)
chunks = list(chunk_iter)

Token indices sequence length is longer than the specified maximum sequence length for this model (1523 > 512). Running this sequence through the model will result in indexing errors


In [20]:
from typing import Iterable, Optional

from docling_core.transforms.chunker.base import BaseChunk
from docling_core.transforms.chunker.hierarchical_chunker import DocChunk
from docling_core.types.doc.labels import DocItemLabel
from rich.console import Console
from rich.panel import Panel

console = Console(
    width=200,  # for getting Markdown tables rendered nicely
)


def find_n_th_chunk_with_label(
    iter: Iterable[BaseChunk], n: int, label: DocItemLabel
) -> Optional[DocChunk]:
    num_found = -1
    for i, chunk in enumerate(iter):
        doc_chunk = DocChunk.model_validate(chunk)
        for it in doc_chunk.meta.doc_items:
            if it.label == label:
                num_found += 1
                if num_found == n:
                    return i, chunk
    return None, None


def print_chunk(chunks, chunk_pos):
    chunk = chunks[chunk_pos]
    ctx_text = chunker.contextualize(chunk=chunk)
    num_tokens = tokenizer.count_tokens(text=ctx_text)
    doc_items_refs = [it.self_ref for it in chunk.meta.doc_items]
    title = f"{chunk_pos=} {num_tokens=} {doc_items_refs=}"
    console.print(Panel(ctx_text, title=title))

In [21]:
i, chunk = find_n_th_chunk_with_label(chunks, n=1, label=DocItemLabel.TABLE)
print_chunk(
    chunks=chunks,
    chunk_pos=i,
)


╭───────────────────────────────────────────────────────────────────── chunk_pos=19 num_tokens=499 doc_items_refs=['#/tables/0'] ──────────────────────────────────────────────────────────────────────╮
│ 4.2 EXPERIMENTAL RESULTS                                                                                                                                                                             │
│ Single-agent System, GSM8K = Single-agent System. Single-agent System, MATH = Single-agent System. Single-agent System, MBPP = Single-agent System. Single-agent System, HumanEval = Single-agent    │
│ System. Single-agent System, Avg. = Single-agent System. IO, GSM8K = 87.8. IO, MATH = 48.6. IO, MBPP = 73.9. IO, HumanEval = 87.0. IO, Avg. = 74.3. CoT (Wei et al., 2023), GSM8K = 87.0. CoT (Wei   │
│ et al., 2023), MATH = 48.8. CoT (Wei et al., 2023), MBPP = 74.2. CoT (Wei et al., 2023), HumanEval = 88.6. CoT (Wei et al., 2023), Avg. = 74.7. CoT SC (Wang et al., 2023b), GSM8K = 86.9. CoT SC    │
│ (Wang et al., 2023b), MATH = 50.4. CoT SC (Wang et al., 2023b), MBPP = 73.3. CoT SC (Wang et al., 2023b), HumanEval = 91.6. CoT SC (Wang et al., 2023b), Avg. = 75.6. Hand-crafted Multi-agent       │
│ System, GSM8K = Hand-crafted Multi-agent System. Hand-crafted Multi-agent System, MATH = Hand-crafted Multi-agent System. Hand-crafted Multi-agent System, MBPP = Hand-crafted Multi-agent System.   │
│ Hand-crafted Multi-agent System, HumanEval = Hand-crafted Multi-agent System. Hand-crafted Multi-agent System, Avg. = Hand-crafted Multi-agent System. SELF-REFINE (Madaan et al., 2023), GSM8K =    │
│ 85.5. SELF-REFINE (Madaan et al., 2023), MATH = 46.1. SELF-REFINE (Madaan et al., 2023), MBPP = 71.8. SELF-REFINE (Madaan et al., 2023), HumanEval = 87.8.                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [22]:
context_chunks = []
threshold = 200

for chunk in chunks:
    text = chunk.__dict__['text']
    if len(text) > threshold:
        ctx_chunk = chunker.contextualize(chunk)
        context_chunks.append(ctx_chunk)

print("Total chunks after filtering:", len(context_chunks))

Total chunks after filtering: 25


In [ ]:
def count_text_stats(text):
    words = text.split()
    word_count = len(words)
    char_count = len(text)
    char_count_no_spaces = len(text.replace(" ", ""))

    return {
        "words": word_count,
        "characters_with_spaces": char_count,
        "characters_without_spaces": char_count_no_spaces
    }


threshold = 30000
small_chunk_count = 0

for i in range(len(context_chunks)):
    text = context_chunks[i]
    stats = count_text_stats(text)
    char_len = stats["characters_with_spaces"]

    if char_len <= threshold:
        small_chunk_count += 1

        print("\n------ Small Chunk Found ------")
        print(f"Chunk Index: {i}")
        print(f"Characters: {char_len}, Words: {stats['words']}")
        print(text)
        print("------ End Chunk ------")

print("\nTotal context_chunks <= 200 characters:", small_chunk_count)

In [ ]:
!pip install sentence-transformers

In [ ]:
# Demonstrating contextual understanding
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
import torch
# 1. Raw text
document = "wuevf hguy uwiebf iuwebf"

# 2. Load the complete model (includes tokenizer)
model = SentenceTransformer('all-MiniLM-L6-v2')

# 3. Access the model's tokenizer for inspection (optional)
tokenizer = model.tokenizer  # This is the model's own tokenizer
tokens = tokenizer.tokenize(document)
print("Tokens:", tokens)

Tokens: ['photos', '##yn', '##thesis', 'is', 'the', 'process', 'by', 'which', 'plants', 'convert', 'sunlight', 'into', 'energy']


In [ ]:
document = "The price is 250 dollars"
tokenizer = model.tokenizer  # This is the model's own tokenizer
tokens = tokenizer.tokenize(document)
print("Tokens:", tokens)
token_ids = tokenizer.encode(document)
print("Token IDs:", token_ids)

Tokens: ['the', 'price', 'is', '250', 'dollars']
Token IDs: [101, 1996, 3976, 2003, 5539, 6363, 102]


In [ ]:
#IMP: This line loads ONLY the tokenizer component
token_ids = tokenizer.encode(document)
print("Token IDs:", token_ids)
# [101, 6302, 23433, 21369, 2003, 1996, 2832, ...]

Token IDs: [101, 2026, 2171, 2003, 7592, 102]


In [ ]:
embedding = model.encode(document)
print(f"Embedding shape: {embedding.shape}")
# Embedding shape: (384,)

Embedding shape: (384,)


In [ ]:
from pdfminer.converter import HTMLConverter
from pdfminer.layout import LAParams
from pdfminer.pdfinterp import PDFPageInterpreter, PDFResourceManager
from pdfminer.pdfpage import PDFPage

input_pdf = "c:\\Users\\hp\\OneDrive\\Desktop\\Rag-Chatbot\\Vanilla_document_based_rag\\JudgeFlow.pdf"
output_html = "c:\\Users\\hp\\OneDrive\\Desktop\\Rag_bot\\JudgeFlow.html"

resource_manager = PDFResourceManager()
laparams = LAParams()

with open(input_pdf, "rb") as pdf_file, open(output_html, "wb") as html_file:
    
    converter = HTMLConverter(resource_manager, html_file, laparams=laparams)
    interpreter = PDFPageInterpreter(resource_manager, converter)

    for page in PDFPage.get_pages(pdf_file):
        interpreter.process_page(page)

    converter.close()

In [ ]:
import sys
!(sys.executable) -m pip install lxml

-m was unexpected at this time.


In [ ]:
from bs4 import BeautifulSoup

with open(r"C:\\Users\\hp\\OneDrive\\Desktop\\rag_bot\\JudgeFlow.html", "r", encoding="utf-8") as f:
    html = f.read()

soup = BeautifulSoup(html, "lxml")

In [ ]:
import re

for span in soup.find_all("span"):
    style = span.get("style", "")

    match = re.search(r"font-size:(\d+\.?\d*)px", style)

    if match:
        size = float(match.group(1))

        if size > 15:
            print("Heading:", span.text)

Heading: J
Heading: F
Heading: : A
Heading: W
Heading: O
Heading: B
Heading: J


In [ ]:
import sys
%pip install pymupdf pdfplumber reportlab

  Using cached pdfplumber-0.11.9-py3-none-any.whl.metadata (43 kB)
  Using cached reportlab-4.4.10-py3-none-any.whl.metadata (1.7 kB)
  Using cached pdfminer_six-20251230-py3-none-any.whl.metadata (4.3 kB)
Using cached pdfplumber-0.11.9-py3-none-any.whl (60 kB)
Using cached pdfminer_six-20251230-py3-none-any.whl (6.6 MB)
Using cached reportlab-4.4.10-py3-none-any.whl (2.0 MB)
  Attempting uninstall: pdfminer.six
    Found existing installation: pdfminer.six 20260107
    Uninstalling pdfminer.six-20260107:
      Successfully uninstalled pdfminer.six-20260107
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
#import fitz 
import pdfplumber
from collections import Counter
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import SimpleDocTemplate, Preformatted



def extract_header_fontsize_from_pdf(pdf_path):
    font_size_counter = Counter()

    with pdfplumber.open(pdf_path) as pdf:
        pages_consider = min(10, len(pdf.pages))  # Consider only the first 10 pages for font size analysis
        for i in range(pages_consider):
            # lines1 = pdf.pages[i].extract_text().split('\n')
            words = pdf.pages[i].extract_words(extra_attrs=['fontname', 'size'])
            lines = {}

            for word in words:
                line_num = word['top']
                if line_num not in lines:
                    lines[line_num] = []
                lines[line_num].append(word)

            for line_words in lines.values():
                font_size_counter[line_words[0]['size']] += 1

    # Find the font sizes that were used more than once
    repeated_sizes = [size for size, count in font_size_counter.items() if count >= 1]
    print(repeated_sizes)
    # Return the highest font size among the repeated sizes
    if repeated_sizes:
        return max(repeated_sizes)
    else:
        return None

In [ ]:
extracted_font_size = extract_header_fontsize_from_pdf(input_pdf)
print(f"Extracted header font size: {extracted_font_size}")

[17.21539999999993, 13.772299999999973, 9.962599999999952, 6.973799999999983, 9.962600000000066, 11.95519999999999, 9.56409999999994, 7.970100000000002, 9.962600000000009, 9.564099999999996, 9.962600000000037, 9.96259999999998, 9.962599999999995, 5.977599999999995, 8.966400000000007, 9.962600000000002, 8.879999999999995, 6.659999999999997, 14.439999999999998, 5.560000000000002, 10.0, 5.0, 6.660000000000025, 7.779999999999973, 7.970100000000059, 6.738760000000013, 9.626800000000003, 8.664120000000025, 8.664119999999969, 5.776080000000036, 8.966400000000021, 8.966399999999993, 7.970099999999988, 11.955200000000005, 9.56410000000001, 8.669340000000034, 5.779560000000004, 6.742819999999995, 6.973800000000011, 6.973799999999997, 8.445700000000102, 6.756559999999922, 8.445699999999988, 4.0539360000000215, 5.91199000000006, 5.06741999999997, 8.966399999999908, 7.17310000000009, 4.981299999999976, 4.98129999999999, 4.9813000000000045, 4.981299999999919, 4.981300000000033, 8.966399999999965, 6.

In [ ]:
def extract_lines_with_font_size(pdf_path, target_font_size):
    lines_with_target_font_size = []

    with pdfplumber.open(pdf_path) as pdf:
        for i in range(len(pdf.pages)):
            words = pdf.pages[i].extract_words(extra_attrs=['fontname', 'size'])
            lines = {}

            for word in words:
                line_num = word['top']
                if line_num not in lines:
                    lines[line_num] = []
                lines[line_num].append(word)

            for line_num, line_words in lines.items():
                line_font_sizes = [word['size'] for word in line_words]
                if target_font_size in line_font_sizes:
                    line_text = ' '.join([word['text'] for word in line_words])
                    lines_with_target_font_size.append(line_text)

    return lines_with_target_font_size

In [ ]:
lines = extract_lines_with_font_size(r"c:\\Users\\hp\\OneDrive\\Desktop\\Rag-Chatbot\\Vanilla_document_based_rag\\JudgeFlow.pdf", 9.962599999999952)

In [ ]:
for i in lines:
    print(i)

ZihanMa ZhikaiZhao ChuanboHua FedericoBerto JinkyooPark
problematicblocks. Thesefine-graineddiagnosticsignalsarethenleveragedby
blockintheworkflow. Ourapproachimprovessampleefficiency,enhancesinter-
automatingincreasinglycomplexagenticworkflows. WeevaluateJ F
achievessuperiorperformanceandefficiencycomparedtoexistingmethods. The
Large language models (LLMs) (Brown et al., 2020) have achieved remarkable success across a
byintegratingLLMsintointelligentagentarchitectures,theemergingfoundationagents(Liuetal.,
2025a) have attracted more attention. Starting from early work on prompt engineering, such as
developmentsinmulti-agentsystemapproaches(Duetal.,2023;Lietal.,2023;Hongetal.,2024),
mathematicalreasoning(Cobbeetal.,2021), codegeneration(Austinetal.,2021), andquestion
if-else
Toaddressthesechallenges,weintroduceJ F ,anEvaluation-Judge-Optimization-Update
blockscapturethreefundamentalformsoflogic: sequential,loop,andconditional,whichareableto
operations or functionalities (Zhang et al., 2

In [ ]:
import sys
print(sys.executable)

c:\Users\hp\OneDrive\Desktop\rag_bot\.venv\Scripts\python.exe
